In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier


In [2]:
train_raw = pd.read_csv("train.csv")
train_soft_label_raw = pd.read_csv("train_soft_label.csv")
test_raw = pd.read_csv("test.csv")
edges_raw = pd.read_csv("txs_edgelist.csv")

print("train_raw:", train_raw.shape)
print("train_soft_label_raw:", train_soft_label_raw.shape)
print("test_raw:", test_raw.shape)
print("edges_raw:", edges_raw.shape)
print("train columns:", train_raw.columns.tolist()[:10], "...")
print("soft-label columns:", train_soft_label_raw.columns.tolist()[-10:])


train_raw: (141772, 168)
train_soft_label_raw: (141772, 170)
test_raw: (15329, 168)
edges_raw: (234355, 2)
train columns: ['txId', 'time_step', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8'] ...
soft-label columns: ['feat_159', 'feat_160', 'feat_161', 'feat_162', 'feat_163', 'feat_164', 'feat_165', 'label', 'p_illicit', 'label_source']


In [3]:
SOFT_LABEL_WEIGHT = 0.5
CONFIDENCE_CUTOFF = 0.7
RECENCY_DECAY = 0.03
USE_GRAPH_FEATURES = True
USE_TIME_STEP = False

SUBMISSION_PATH = "attempt5_tuned_LGBM.csv"

best_lightgbm_params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 15,
    "max_depth": 10,
    "min_child_samples": 20,
    "min_split_gain": 0.1,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}


In [4]:
node_times = pd.concat(
    [
        train_raw[["txId", "time_step"]],
        test_raw[["txId", "time_step"]],
    ],
    ignore_index=True,
).drop_duplicates("txId").set_index("txId")["time_step"]

edges = edges_raw[["txId1", "txId2"]].copy()
edges["time_1"] = edges["txId1"].map(node_times)
edges["time_2"] = edges["txId2"].map(node_times)

graph_feature_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "degree_ratio",
]

def temporal_node_features(cutoff_step):
    usable_edges = edges[
        edges["time_1"].notna()
        & edges["time_2"].notna()
        & (edges["time_1"] <= cutoff_step)
        & (edges["time_2"] <= cutoff_step)
    ]

    in_degree = usable_edges.groupby("txId2").size().rename("in_degree")
    out_degree = usable_edges.groupby("txId1").size().rename("out_degree")

    features = pd.concat([in_degree, out_degree], axis=1).fillna(0)
    features["total_degree"] = (
        features["in_degree"] + features["out_degree"]
    )
    features["degree_ratio"] = (
        features["out_degree"]
        / features["in_degree"].replace(0, np.nan)
    )
    features["degree_ratio"] = (
        features["degree_ratio"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return features.reset_index().rename(columns={"index": "txId"})

def add_temporal_graph_features(frame, cutoff_step):
    graph_features = temporal_node_features(cutoff_step)
    result = frame.merge(graph_features, on="txId", how="left")
    result[graph_feature_cols] = result[graph_feature_cols].fillna(0)
    return result

def calculate_recency_weights(time_steps, reference_time_step, decay_rate):
    age = reference_time_step - time_steps
    return np.exp(-decay_rate * age)


In [5]:
observed_mask = train_soft_label_raw["label"].notna()
soft_label_mask = (
    train_soft_label_raw["label"].isna()
    & train_soft_label_raw["p_illicit"].notna()
)
confidence_mask = (
    (train_soft_label_raw["p_illicit"] >= CONFIDENCE_CUTOFF)
    | (train_soft_label_raw["p_illicit"] <= 1 - CONFIDENCE_CUTOFF)
)
confident_pseudo_mask = soft_label_mask & confidence_mask

observed_rows = train_soft_label_raw.loc[observed_mask].copy()
observed_rows["training_target"] = observed_rows["label"].astype(int)
observed_rows["label_source_weight"] = 1.0

pseudo_rows = train_soft_label_raw.loc[confident_pseudo_mask].copy()
pseudo_rows["training_target"] = (
    pseudo_rows["p_illicit"] >= 0.5
).astype(int)

pseudo_rows["label_source_weight"] = SOFT_LABEL_WEIGHT

final_train = pd.concat(
    [observed_rows, pseudo_rows],
    ignore_index=True,
)

print("Confirmed rows:", len(observed_rows))
print("Confident soft-labelled rows:", len(pseudo_rows))
print("Total final training rows:", len(final_train))


Confirmed rows: 31235
Confident soft-labelled rows: 105509
Total final training rows: 136744


In [6]:
if USE_GRAPH_FEATURES:
    final_train = add_temporal_graph_features(
        final_train,
        cutoff_step=int(final_train["time_step"].max()),
    )

test_features = []
for step, step_rows in test_raw.groupby("time_step", sort=True):
    step_rows = step_rows.copy()
    if USE_GRAPH_FEATURES:
        step_rows = add_temporal_graph_features(
            step_rows,
            cutoff_step=int(step),
        )
    test_features.append(step_rows)

test_features = pd.concat(test_features, ignore_index=True)

feature_cols = [
    c for c in train_raw.columns
    if c not in ["index", "txId", "label", "time_step"]
]

final_feature_cols = list(feature_cols)
if USE_GRAPH_FEATURES:
    final_feature_cols += graph_feature_cols
if USE_TIME_STEP:
    final_feature_cols += ["time_step"]

X_final = final_train[final_feature_cols]
y_final = final_train["training_target"].astype(float)
X_test = test_features[final_feature_cols]

print("Feature count:", len(final_feature_cols))
print("Training shape:", X_final.shape)
print("Test shape:", X_test.shape)


Feature count: 169
Training shape: (136744, 169)
Test shape: (15329, 169)


In [7]:
# Hard versions are used only to calculate class-balance weights.
# The actual training target remains the continuous soft target.
weighting_labels = (
    final_train["training_target"] >= 0.5
).astype(int)

class_weights = compute_sample_weight(
    class_weight="balanced",
    y=weighting_labels,
)

recency_weights = calculate_recency_weights(
    final_train["time_step"].to_numpy(),
    reference_time_step=int(final_train["time_step"].max()),
    decay_rate=RECENCY_DECAY,
)

source_weights = final_train["label_source_weight"].to_numpy()

final_sample_weights = (
    class_weights * recency_weights * source_weights
)
final_sample_weights = (
    final_sample_weights / final_sample_weights.mean()
)

print("Sample-weight minimum:", final_sample_weights.min())
print("Sample-weight maximum:", final_sample_weights.max())


Sample-weight minimum: 0.2389010638761486
Sample-weight maximum: 17.480467126166644


In [8]:

final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "model",
        LGBMClassifier(
            objective="binary",
            random_state=100,
            n_jobs=-1,
            verbosity=-1,
            **best_lightgbm_params,
        ),
    ),
])

final_model.fit(
    X_final,
    y_final,
    model__sample_weight=final_sample_weights,
)

print("Final model fitted successfully.")


Final model fitted successfully.


In [9]:
test_predictions = final_model.predict_proba(X_test)[:, 1]

test_predictions = np.clip(
    test_predictions,
    0.0,
    1.0,
)

submission = pd.DataFrame({
    "index": test_raw["index"].to_numpy(),
    "target": test_predictions,
})

submission_path = SUBMISSION_PATH
submission.to_csv(
    submission_path,
    index=False,
)

display(submission.head())

print("Saved:", submission_path)
print(
    "Prediction range:",
    submission["target"].min(),
    "to",
    submission["target"].max(),
)
print(
    "Number of unique predictions:",
    submission["target"].nunique(),
)

,index,target
0,0,0.000278
1,1,0.000723
2,2,0.000890
3,3,0.000421
4,4,0.000017


Saved: attempt5_tuned_LGBM.csv
Prediction range: 1.5993088084540947e-07 to 0.9999790461808454
Number of unique predictions: 15281
